# Calphy Executor Integration Demo

This notebook demonstrates the new executorlib support in phase_diagram_workflows.
It shows both the traditional usage and the new executor-based parallel execution.


In [1]:
import sys
from pathlib import Path

# Add project root to path for imports
PROJECT_ROOT = next(
    (parent for parent in [Path.cwd(), *Path.cwd().parents] if (parent / "pyproject.toml").is_file()),
    Path.cwd(),
)
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from phase_diagram_workflows.calphy import calculator as calphy_calc
from executorlib import SingleNodeExecutor
from pylammpsmpi import LammpsLibrary, init_function

EXAMPLE_ROOT = PROJECT_ROOT / "notebooks" / "Aluminium_free_energy"

In [2]:
import os
from typing import Dict, Any, Optional
from ase.atoms import Atoms
from ase.build import bulk
from lammpsparser import get_potential_by_name
import pandas as pd

## 1. Structure and Potential Setup

In [3]:
# Create Aluminum structure
Al_pure_structure = bulk('Al', cubic=True).repeat(3)
print(f"Created Al structure with {len(Al_pure_structure)} atoms")

Created Al structure with 108 atoms


In [4]:
# Get potential for Al
potential_df = get_potential_by_name('2009--Mendelev-M-I--Al-Mg--LAMMPS--ipr1')
potential_df = potential_df.to_frame().transpose()
print("Potential loaded successfully:")
display(potential_df.head())

Potential loaded successfully:


/cmmc/ptmp/pchilaka/Packages/lammpsparser/src/lammpsparser/potential.py:326: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_pot["Config"] = config_lst


,Config,Filename,Model,Name,Species,Citations
164,"[pair_style eam/fs, pair_coeff * * /cmmc/ptmp/...",[potential_LAMMPS/2009--Mendelev-M-I--Al-Mg--L...,NISTiprpy,2009--Mendelev-M-I--Al-Mg--LAMMPS--ipr1,"[Al, Mg]",[{'Mendelev_2009': {'title': 'Development of i...


## 2. Traditional Usage (Backward Compatible)

This shows the original API that continues to work unchanged.

In [5]:
# Parameters for free energy calculation
input_params_fe = {  
    "mode": "fe", 
    "temperature": 300,  
    "n_equilibration_steps": 2500,
    "n_switching_steps": 2500,
    "n_print_steps": 1000,
    "equilibration_control": "nose-hoover",
    "queue": {
        "cores": 1,  
        "scheduler": "local"  
    },
    'reference_phase': 'solid',
    'file_format': 'lammps-data',
}

In [6]:
# Traditional usage - no executor, no metadata
print("Running traditional calphy calculation...")
calphy_results_traditional = calphy_calc.calc_free_energy_with_calphy(
    input_structure=Al_pure_structure,
    potential_df=potential_df,
    calphy_parameters=input_params_fe,
    working_directory=str(EXAMPLE_ROOT / "Al_traditional"),
)
print("Traditional calculation completed successfully!")

Running traditional calphy calculation...
Input validation successful. Proceeding with calculation.


/cmmc/ptmp/pyironhb/pyiron_latest_env/lib/python3.12/site-packages/ase/io/lammpsdata.py:72: FutureWarning: "style" is deprecated; please use "atom_style".
  warnings.warn(
/cmmc/ptmp/pyironhb/pyiron_latest_env/lib/python3.12/site-packages/pydantic/main.py:463: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Unexpected Value)
  PydanticSerializationUnexpectedValue(Expected `list[float]` - serialized value may not be as expected [input_value=0, input_type=int])
  PydanticSerializationUnexpectedValue(Expected `list[list[float]]` - serialized value may not be as expected [input_value=0, input_type=int])
  return self.__pydantic_serializer__.to_python(


LAMMPS (29 Aug 2024)
OMP_NUM_THREADS environment is not set. Defaulting to 1 thread. (src/comm.cpp:98)
  using 1 OpenMP thread(s) per MPI task

The 'box' command has been removed and will be ignored

Reading data file ...
  orthogonal box = (0 0 0) to (12.15 12.15 12.15)
  1 by 1 by 1 MPI processor grid
  reading atoms ...
  108 atoms
  read_data CPU = 0.004 seconds

CITE-CITE-CITE-CITE-CITE-CITE-CITE-CITE-CITE-CITE-CITE-CITE-CITE

Your simulation uses code contributions which should be cited:
- Type Label Framework: https://doi.org/10.1021/acs.jpcb.3c08419
The log file lists these citations in BibTeX format.

CITE-CITE-CITE-CITE-CITE-CITE-CITE-CITE-CITE-CITE-CITE-CITE-CITE

Neighbor list info ...
  update: every = 1 steps, delay = 0 steps, check = yes
  max neighbors/atom: 2000, page size: 100000
  master list distance cutoff = 9.5
  ghost atom cutoff = 9.5
  binsize = 4.75, bins = 3 3 3
  1 neighbor lists, perpetual/occasional/extra = 1 0 0
  (1) pair eam/fs, perpetual
      attribut

In [7]:
# Show traditional results
print("Calculation object:")
display(calphy_results_traditional[0])
print("\nResults DataFrame:")
display(calphy_results_traditional[1])

Calculation object:


Calculation(monte_carlo=MonteCarlo(n_steps=1, n_swaps=0, forward_swap_types=[], reverse_swap_types=[], allow_all_swaps=True, use_custom_lammps=False), composition_scaling=CompositionScaling(output_chemical_composition={}, restrictions=[]), md=MD(timestep=0.001, n_small_steps=10000, n_every_steps=10, n_repeat_steps=10, n_cycles=100, thermostat_damping=0.1, barostat_damping=0.1, cmdargs='', init_commands=[]), nose_hoover=NoseHoover(thermostat_damping=0.1, barostat_damping=0.1), berendsen=Berendsen(thermostat_damping=100.0, barostat_damping=100.0), queue=Queue(scheduler='local', cores=1, jobname='calphy', walltime='23:59:00', queuename='', memory='3GB', commands=[], options={}), tolerance=Tolerance(lattice_constant=0.0002, spring_constant=0.1, solid_fraction=0.7, liquid_fraction=0.05, pressure=10.0), uhlenbeck_ford_model=UFMP(p=50.0, sigma=1.5), melting_temperature=MeltingTemperature(guess=None, step=200, attempts=5), materials_project=MaterialsProject(api_key='', conventional=True, targe


Results DataFrame:


,calculation_mode,status,temperature,pressure,free_energy,reference_phase,error_code,composition,calculation,ideal_entropy,reference_composition,Al,Mg
0,fe,True,300.0,0,-3.422605,solid,None,"{'Al': 1.0, 'Mg': 0.0}",Al_traditional,0,0.0,1.0,0.0


## 3. New Executor-Based Usage

This demonstrates the new executorlib integration with parallel execution.

In [8]:
# Set up executor and LAMMPS library
print("Setting up executor...")
executor = SingleNodeExecutor(
    block_allocation=True,
    hostname_localhost=True,
    max_workers=1,
    init_function=init_function,
    resource_dict={
        "cores": 1,
        "cwd": str(EXAMPLE_ROOT / "Al_executor"),
    },
)

# Create LAMMPS library with executor
lmp = LammpsLibrary(cores=1, executor=executor)
print("Executor and LAMMPS library ready!")

Setting up executor...
LAMMPS (29 Aug 2024)
OMP_NUM_THREADS environment is not set. Defaulting to 1 thread. (src/comm.cpp:98)
  using 1 OpenMP thread(s) per MPI task
Executor and LAMMPS library ready!


In [9]:
# New usage with executor and metadata
print("Running calphy calculation with executor...")
metadata_dict = {
    'project': 'calphy-executor-demo',
    'material': 'Aluminum',
    'temperature': 300,
    'method': 'executor-based',
    'version': '1.0'
}

calphy_results_executor = calphy_calc.calc_free_energy_with_calphy(
    input_structure=Al_pure_structure,
    potential_df=potential_df,
    calphy_parameters=input_params_fe,
    working_directory=str(EXAMPLE_ROOT / "Al_executor"),
    lmp=lmp,  # Pass the LAMMPS library with executor
    metadata_dict=metadata_dict  # Pass metadata for caching
)
print("Executor-based calculation completed successfully!")

Running calphy calculation with executor...
Input validation successful. Proceeding with calculation.


/cmmc/ptmp/pyironhb/pyiron_latest_env/lib/python3.12/site-packages/ase/io/lammpsdata.py:72: FutureWarning: "style" is deprecated; please use "atom_style".
  warnings.warn(
/cmmc/ptmp/pyironhb/pyiron_latest_env/lib/python3.12/site-packages/pydantic/main.py:463: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Unexpected Value)
  PydanticSerializationUnexpectedValue(Expected `list[float]` - serialized value may not be as expected [input_value=0, input_type=int])
  PydanticSerializationUnexpectedValue(Expected `list[list[float]]` - serialized value may not be as expected [input_value=0, input_type=int])
  return self.__pydantic_serializer__.to_python(



The 'box' command has been removed and will be ignored

Reading data file ...
  orthogonal box = (0 0 0) to (12.15 12.15 12.15)
  1 by 1 by 1 MPI processor grid
  reading atoms ...
  108 atoms
  read_data CPU = 0.003 seconds

CITE-CITE-CITE-CITE-CITE-CITE-CITE-CITE-CITE-CITE-CITE-CITE-CITE

Your simulation uses code contributions which should be cited:
- Type Label Framework: https://doi.org/10.1021/acs.jpcb.3c08419
The log file lists these citations in BibTeX format.

CITE-CITE-CITE-CITE-CITE-CITE-CITE-CITE-CITE-CITE-CITE-CITE-CITE

Neighbor list info ...
  update: every = 1 steps, delay = 0 steps, check = yes
  max neighbors/atom: 2000, page size: 100000
  master list distance cutoff = 9.5
  ghost atom cutoff = 9.5
  binsize = 4.75, bins = 3 3 3
  1 neighbor lists, perpetual/occasional/extra = 1 0 0
  (1) pair eam/fs, perpetual
      attributes: half, newton on
      pair build: half/bin/atomonly/newton
      stencil: half/bin/3d
      bin: standard
Setting up Verlet run ...
  Unit 

In [10]:
# Show executor results
print("Calculation object:")
display(calphy_results_executor[0])
print("\nResults DataFrame:")
display(calphy_results_executor[1])
print("\nMetadata used:")
print(metadata_dict)

Calculation object:


Calculation(monte_carlo=MonteCarlo(n_steps=1, n_swaps=0, forward_swap_types=[], reverse_swap_types=[], allow_all_swaps=True, use_custom_lammps=False), composition_scaling=CompositionScaling(output_chemical_composition={}, restrictions=[]), md=MD(timestep=0.001, n_small_steps=10000, n_every_steps=10, n_repeat_steps=10, n_cycles=100, thermostat_damping=0.1, barostat_damping=0.1, cmdargs='', init_commands=[]), nose_hoover=NoseHoover(thermostat_damping=0.1, barostat_damping=0.1), berendsen=Berendsen(thermostat_damping=100.0, barostat_damping=100.0), queue=Queue(scheduler='local', cores=1, jobname='calphy', walltime='23:59:00', queuename='', memory='3GB', commands=[], options={}), tolerance=Tolerance(lattice_constant=0.0002, spring_constant=0.1, solid_fraction=0.7, liquid_fraction=0.05, pressure=10.0), uhlenbeck_ford_model=UFMP(p=50.0, sigma=1.5), melting_temperature=MeltingTemperature(guess=None, step=200, attempts=5), materials_project=MaterialsProject(api_key='', conventional=True, targe


Results DataFrame:


,calculation_mode,status,temperature,pressure,free_energy,reference_phase,error_code,composition,calculation,ideal_entropy,reference_composition,Al,Mg
0,fe,True,300.0,0,-3.422605,solid,None,"{'Al': 1.0, 'Mg': 0.0}",Al_traditional,0,0.0,1.0,0.0
1,fe,True,300.0,0,-3.423374,solid,None,"{'Al': 1.0, 'Mg': 0.0}",Al_executor,0,0.0,1.0,0.0



Metadata used:
{'project': 'calphy-executor-demo', 'material': 'Aluminum', 'temperature': 300, 'method': 'executor-based', 'version': '1.0'}


## 4. Comparison and Benefits

### Key Differences:
- **Traditional**: Runs sequentially, no parallel execution
- **Executor-based**: Runs with parallel execution via executorlib

### Benefits of Executor Integration:
- ✅ Parallel execution for faster calculations
- ✅ Resource management via executorlib
- ✅ Metadata support for result caching and retrieval
- ✅ Better resource utilization
- ✅ Full backward compatibility maintained

In [11]:
# Clean up executor
if 'executor' in locals():
    executor.shutdown()
    print("Executor shutdown complete")

print("Demo completed successfully! 🎉")

Total wall time: 0:01:04
Executor shutdown complete
Demo completed successfully! 🎉
